# 08 · Executive Business Summary

**Goal:** bring the headline numbers together in one place and link each analysis to a business decision.

**Source:** `gold.report_customers`, `gold.report_products`, `gold.fact_sales`

**Note on AOV:** here AOV = total revenue ÷ total orders. Notebook 06 reports the mean of customer-level AOV, which is a different number.

**Output:** KPI table, segment and product snapshots, and the business-question map in `output/`

## 1. Setup

In [2]:
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
import config

os.makedirs("output", exist_ok=True)
engine = create_engine(config.get_conn_url())

## 2. Load data

In [11]:
customers = pd.read_sql("SELECT * FROM gold.report_customers", engine)
products = pd.read_sql("SELECT * FROM gold.report_products", engine)
sales = pd.read_sql(
    """
    SELECT order_number, order_date, customer_key, product_key, sales_amount, quantity
    FROM gold.fact_sales
    WHERE order_date IS NOT NULL
    """,
    engine,
)
sales["order_date"] = pd.to_datetime(sales["order_date"])
ANALYSIS_DATE = sales["order_date"].max()

## 3. Business KPIs

In [12]:
kpis = {
    "total_customers": customers["customer_key"].nunique(),
    "total_revenue": sales["sales_amount"].sum(),
    "total_orders": sales["order_number"].nunique(),
    "total_units": sales["quantity"].sum(),
    "products": products["product_key"].nunique(),
    "analysis_start": sales["order_date"].min().date(),
    "analysis_end": ANALYSIS_DATE.date(),
}

orders_per_customer = kpis["total_orders"] / kpis["total_customers"]
aov = kpis["total_revenue"] / kpis["total_orders"]

kpi_table = pd.DataFrame(
    [
        ["Total Customers", kpis["total_customers"]],
        ["Total Revenue", kpis["total_revenue"]],
        ["Total Orders", kpis["total_orders"]],
        ["Units Sold", kpis["total_units"]],
        ["Unique Products", kpis["products"]],
        ["Orders per Customer", orders_per_customer],
        ["AOV", aov],
    ],
    columns=["KPI", "Value"],
)

print(kpi_table)

                   KPI         Value
0      Total Customers  1.848200e+04
1        Total Revenue  2.935126e+07
2         Total Orders  2.765700e+04
3           Units Sold  6.040400e+04
4      Unique Products  1.300000e+02
5  Orders per Customer  1.496429e+00
6                  AOV  1.061260e+03


18,482 customers · revenue 29,351,258 · 27,657 orders · 60,404 units · 130 products · 1.50 orders per customer · AOV 1,061.26.

## 4. Customer segment table

In [17]:
segment_table = (
    customers.groupby("customer_segments")
    .agg(
        customers=("customer_key", "nunique"),
        revenue=("total_sales", "sum"),
        avg_aov=("avg_order_value", "mean"),
    )
    .sort_values("revenue", ascending=False)
)
segment_table["revenue_share_%"] = (
    100 * segment_table["revenue"] / segment_table["revenue"].sum()
)
segment_table["revenue_per_customer"] = (
    segment_table["revenue"] / segment_table["customers"]
)
print(segment_table.round(2).reset_index())

  customer_segments  customers   revenue  avg_aov  revenue_share_%  \
0               new      14629  11086797   601.24            37.77   
1               vip       1653  10760470  2633.23            36.66   
2           regular       2200   7503991  1682.19            25.57   

   revenue_per_customer  
0                757.86  
1               6509.66  
2               3410.90  


VIP customers are 8.9% of customers (1,653 of 18,482) and generate 36.7% of revenue. New customers are 79.2% of customers and generate 37.8%.

## 5. Product portfolio snapshot (by category)

In [19]:
product_table = (
    products.groupby("category")
    .agg(
        revenue=("total_sales", "sum"),
        units=("total_quantity", "sum"),
        products=("product_key", "nunique"),
    )
    .sort_values("revenue", ascending=False)
)
product_table["revenue_share_%"] = (
    100 * product_table["revenue"] / product_table["revenue"].sum()
)
print(product_table.round(2))

              revenue  units  products  revenue_share_%
category                                               
Bikes        28316272  15205        88            96.46
Accessories    700262  36112        22             2.39
Clothing       339716   9106        20             1.16


Bikes: 88 of 130 products and 96.5% of revenue. Accessories sell the most units (36,112) but bring 2.4% of revenue.

## 6. From insights to decisions

Each business question is linked to the analysis that answers it and the decision it can support. The KPI and snapshot tables are saved at the end of this cell.

In [21]:
business_questions = pd.DataFrame(
    [
        [
            "Who drives revenue?",
            "RFM / customer segments",
            "Retention + cross-sell focus",
        ],
        [
            "Do customers return?",
            "Cohort retention",
            "Improve post-purchase engagement",
        ],
        [
            "Which products matter?",
            "Pareto + estimated margin",
            "Inventory / merchandising prioritization",
        ],
        [
            "Who may leave?",
            "Future churn model",
            "Prioritize high-risk/high-value customers",
        ],
        [
            "How is sales moving?",
            "Monthly trend + forecast",
            "Planning and target-setting",
        ],
    ],
    columns=["Business Question", "Analysis", "Possible Decision"],
)
print(business_questions)
business_questions.to_csv("output/business_question_to_decision_map.csv", index=False)
kpi_table.to_csv("output/executive_kpi_table.csv", index=False)
segment_table.to_csv("output/executive_segment_snapshot.csv")
product_table.to_csv("output/executive_product_snapshot.csv")

        Business Question                   Analysis  \
0     Who drives revenue?    RFM / customer segments   
1    Do customers return?           Cohort retention   
2  Which products matter?  Pareto + estimated margin   
3          Who may leave?         Future churn model   
4    How is sales moving?   Monthly trend + forecast   

                           Possible Decision  
0               Retention + cross-sell focus  
1           Improve post-purchase engagement  
2   Inventory / merchandising prioritization  
3  Prioritize high-risk/high-value customers  
4                Planning and target-setting  
